In [1]:
# Find the project root regardless of where Jupyter's working directory is set
# (works whether VS Code launches the kernel from notebooks/ or the project root).
import os
from pathlib import Path

def find_project_root(marker="run_pipeline.py"):
    p = Path.cwd()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

ROOT = find_project_root()
os.chdir(ROOT)
print("Working directory set to:", ROOT)

Working directory set to: c:\Users\meghna.ashok\Downloads\clinic-dna-local\next_step_analysis


# 03 — Compare Current Run to Baseline

**Run this after every change** to see exactly what it affected — the "did I
break anything?" check. Compares `output/recommendations_local.csv` (your
latest run) against `baseline/recommendations_baseline.csv` (the snapshot you
froze with `scripts/save_baseline.py`).

**Workflow:**
```
python run_pipeline.py             # baseline run, looks right
python scripts/save_baseline.py    # freeze it
... make a change to a model or the runner ...
python run_pipeline.py             # produces a new output/
# open this notebook -> see exactly what changed
```

If nothing has changed, every section below should show 0 differences.

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None); pd.set_option('display.width', 160)

CURRENT  = "output/recommendations_local.csv"
BASELINE = "baseline/recommendations_baseline.csv"

cur = pd.read_csv(CURRENT, low_memory=False)
base = pd.read_csv(BASELINE, low_memory=False)
cur.columns = [c.strip() for c in cur.columns]
base.columns = [c.strip() for c in base.columns]

import json
manifest = json.load(open("baseline/manifest.json"))
print(f"Baseline saved: {manifest['saved_at']}")
print(f"CURRENT:  {cur.shape[0]:,} rows")
print(f"BASELINE: {base.shape[0]:,} rows")
print(f"Same columns: {sorted(cur.columns) == sorted(base.columns)}")

Baseline saved: 2026-09-05T02:05:48.507821+00:00
CURRENT:  37,830 rows
BASELINE: 37,830 rows
Same columns: True


## 1. Row-level diff

Match rows on the natural key (clinic + category + species) and see what
changed per row: added, removed, or a value changed.

In [3]:
KEY = ['CLINIC ID', 'DISEASE CATEGORY', 'SPECIES']

cur_k  = cur.set_index(KEY)
base_k = base.set_index(KEY)

added   = cur_k.index.difference(base_k.index)
removed = base_k.index.difference(cur_k.index)
common  = cur_k.index.intersection(base_k.index)

print(f"Rows added (new clinic/category/species combos):   {len(added):,}")
print(f"Rows removed (existed in baseline, gone now):        {len(removed):,}")
print(f"Rows present in both (comparable):                   {len(common):,}")

Rows added (new clinic/category/species combos):   0
Rows removed (existed in baseline, gone now):        0
Rows present in both (comparable):                   37,830


## 2. What changed, column by column (among rows present in both)

In [4]:
COMPARE_COLS = [
    'ACTION RECOMMENDATION', 'HAS SAMPLE', 'DISEASE_CATEGORY_EDUCATION_FLAG',
    'DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION',
    'MONTHLY OPPORTUNITY', 'MONTHLY OPPORTUNITY VOLUME',
    'OPPORTUNITY RANKING WITHIN CLINIC', 'CLINIC TIER', 'CLUSTER',
]
COMPARE_COLS = [c for c in COMPARE_COLS if c in cur.columns and c in base.columns]

c_common = cur_k.loc[common, COMPARE_COLS]
b_common = base_k.loc[common, COMPARE_COLS]

print(f"{'column':45s} {'changed rows':>14s} {'% of common':>12s}")
for col in COMPARE_COLS:
    # numeric cols: flag if the value moved at all; strings: exact mismatch
    if pd.api.types.is_numeric_dtype(c_common[col]) or pd.api.types.is_numeric_dtype(b_common[col]):
        cv = pd.to_numeric(c_common[col], errors='coerce')
        bv = pd.to_numeric(b_common[col], errors='coerce')
        diff = (cv.round(2) != bv.round(2)) & ~(cv.isna() & bv.isna())
    else:
        diff = c_common[col].astype(str) != b_common[col].astype(str)
    n = diff.sum()
    print(f"{col:45s} {n:>14,} {n/len(common)*100:>11.2f}%")

column                                          changed rows  % of common
ACTION RECOMMENDATION                                      4        0.01%
HAS SAMPLE                                                 0        0.00%
DISEASE_CATEGORY_EDUCATION_FLAG                            4        0.01%
DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION                4        0.01%
MONTHLY OPPORTUNITY                                        0        0.00%
MONTHLY OPPORTUNITY VOLUME                                 0        0.00%
OPPORTUNITY RANKING WITHIN CLINIC                          0        0.00%
CLINIC TIER                                                0        0.00%
CLUSTER                                                    0        0.00%


## 3. Distribution shift — opportunity & the top-seller targeting metric

Not just "did values change" but "did they change in a way that matters".

In [5]:
def opp_stats(df, label):
    o = pd.to_numeric(df['MONTHLY OPPORTUNITY'], errors='coerce')
    print(f"{label:10s} mean {o.mean():8.2f}  median {o.median():6.2f}  max {o.max():10.2f}  zero% {(o==0).mean()*100:5.1f}%")

opp_stats(base, "BASELINE")
opp_stats(cur,  "CURRENT")


# mean/median/max/zero-share of MONTHLY OPPORTUNITY for baseline vs current — so you catch a shift in the shape of the distribution, not just that individual values moved

BASELINE   mean    30.68  median   0.00  max    8750.21  zero%  66.8%
CURRENT    mean    30.68  median   0.00  max    8750.21  zero%  66.8%


In [6]:
def top_seller_rate(df):
    rank1 = df[df['OPPORTUNITY RANKING WITHIN CLINIC']==1][['CLINIC ID','DISEASE CATEGORY','SPECIES']]
    top_share = (df.sort_values('DISEASE_CATEGORY_SHARE_PCT', ascending=False)
                   .groupby('CLINIC ID').first()[['DISEASE CATEGORY','SPECIES']]
                   .rename(columns={'DISEASE CATEGORY':'top_share_dc','SPECIES':'top_share_sp'}))
    chk = rank1.merge(top_share, on='CLINIC ID')
    chk['same'] = (chk['DISEASE CATEGORY']==chk['top_share_dc']) & (chk['SPECIES']==chk['top_share_sp'])
    return chk['same'].mean()*100

print(f"BASELINE top-seller-is-#1-rec rate: {top_seller_rate(base):.1f}%")
print(f"CURRENT  top-seller-is-#1-rec rate: {top_seller_rate(cur):.1f}%")
print("(Lower is generally better — more gap-finding, less stating the obvious.")
print(" A big jump either way is worth investigating.)")

# how often the #1-ranked recommendation in a clinic is also that clinic's top-selling category (highest DISEASE_CATEGORY_SHARE_PCT).

BASELINE top-seller-is-#1-rec rate: 10.4%
CURRENT  top-seller-is-#1-rec rate: 10.4%
(Lower is generally better — more gap-finding, less stating the obvious.
 A big jump either way is worth investigating.)


## 4. The caps — should almost NEVER change unless you edited them on purpose

In [7]:
def cap_check(df, label):
    sampled = df[df['HAS SAMPLE']==True].groupby('CLINIC SALES REP OR TM')['CLINIC ID'].nunique()
    edu = df[df['DISEASE_CATEGORY_EDUCATION_FLAG']==1]
    print(f"{label:10s} max samples/TM: {sampled.max() if len(sampled) else 0:>3}  "
          f"(cap=14)   |   education recs: {len(edu):>5,}")

cap_check(base, "BASELINE")
cap_check(cur,  "CURRENT")



# Sanity check on the guardrails: max distinct clinics any TM is sampling into (against a hard cap of 14) and the count of education-flagged recs. These are expected to be byte-identical across runs unless you deliberately changed a threshold.

BASELINE   max samples/TM:  14  (cap=14)   |   education recs:   180
CURRENT    max samples/TM:  14  (cap=14)   |   education recs:   180


## 5. Example rows that changed

A handful of actual before/after rows, so you can eyeball whether the change
makes sense — not just that *a* number moved.

In [8]:
if len(common) > 0 and COMPARE_COLS:
    any_diff = pd.Series(False, index=common)
    for col in COMPARE_COLS:
        if pd.api.types.is_numeric_dtype(c_common[col]) or pd.api.types.is_numeric_dtype(b_common[col]):
            cv = pd.to_numeric(c_common[col], errors='coerce'); bv = pd.to_numeric(b_common[col], errors='coerce')
            any_diff |= (cv.round(2) != bv.round(2)) & ~(cv.isna() & bv.isna())
        else:
            any_diff |= c_common[col].astype(str) != b_common[col].astype(str)

    changed_keys = common[any_diff][:5]
    for k in changed_keys:
        print("Clinic/category/species:", k)
        print("  BASELINE:", b_common.loc[k, COMPARE_COLS].to_dict())
        print("  CURRENT: ", c_common.loc[k, COMPARE_COLS].to_dict())
        print()
else:
    print("No comparable rows or columns to show.")

Clinic/category/species: (763348, 'URINARY', 'CAT')
  BASELINE: {'ACTION RECOMMENDATION': 'FOLLOW-UP + SAMPLE', 'HAS SAMPLE': True, 'DISEASE_CATEGORY_EDUCATION_FLAG': 0, 'DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION': 'Sample - Low Penetration Opportunity', 'MONTHLY OPPORTUNITY': 1081.3, 'MONTHLY OPPORTUNITY VOLUME': 0.0, 'OPPORTUNITY RANKING WITHIN CLINIC': 1, 'CLINIC TIER': 'C', 'CLUSTER': 'Outliers'}
  CURRENT:  {'ACTION RECOMMENDATION': 'EDUCATION + SAMPLE', 'HAS SAMPLE': True, 'DISEASE_CATEGORY_EDUCATION_FLAG': 1, 'DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION': 'Education + Sample - Low Penetration Opportunity', 'MONTHLY OPPORTUNITY': 1081.3, 'MONTHLY OPPORTUNITY VOLUME': 0.0, 'OPPORTUNITY RANKING WITHIN CLINIC': 1, 'CLINIC TIER': 'C', 'CLUSTER': 'Outliers'}

Clinic/category/species: (795440, 'GI', 'DOG')
  BASELINE: {'ACTION RECOMMENDATION': 'EDUCATION', 'HAS SAMPLE': False, 'DISEASE_CATEGORY_EDUCATION_FLAG': 1, 'DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION': 'Education - Growth Opp

## 6. WHO did this change affect? (scenario & tier breakdown)

Aggregate change-counts tell you *how much* moved; this tells you *which kinds of clinics* moved. 

In [9]:
# Rows whose MONTHLY OPPORTUNITY moved, broken down by scenario and tier.
SC = 'SCENARIO - CLINIC DISEASE CATEGORY'
merged = base.merge(cur, on=KEY, suffixes=('_base', '_cur'))
ob = pd.to_numeric(merged['MONTHLY OPPORTUNITY_base'], errors='coerce').fillna(0).round(2)
oc = pd.to_numeric(merged['MONTHLY OPPORTUNITY_cur'],  errors='coerce').fillna(0).round(2)
merged['moved'] = ob != oc
merged['delta'] = oc - ob

n_moved = int(merged['moved'].sum())
print(f"Rows where MONTHLY OPPORTUNITY moved: {n_moved:,} of {len(merged):,} ({n_moved/len(merged)*100:.1f}%)")
if n_moved:
    print(f"  went UP: {(merged['delta']>0).sum():,}   went DOWN: {(merged['delta']<0).sum():,}")

if n_moved and (SC + '_base') in merged.columns:
    scen = SC + '_base'
    # over/under-representation: share of moved rows vs share of all rows, per scenario
    moved_mix = merged.loc[merged['moved'], scen].value_counts(normalize=True)
    all_mix   = merged[scen].value_counts(normalize=True)
    tab = pd.DataFrame({
        'moved_%': (moved_mix*100).round(1),
        'overall_%': (all_mix*100).round(1),
    }).fillna(0)
    tab['over_represented_x'] = (tab['moved_%'] / tab['overall_%'].replace(0, float('nan'))).round(1)
    print("\nWhich SCENARIOS moved (over_represented_x >1 = this change hits them more than their share):")
    display(tab.sort_values('over_represented_x', ascending=False))


    # A ratio >1 means the change hit that scenario more than its baseline weight — the signal that a targeted fix landed where intended. Cell 16 repeats it by tier and prints an auto-summary naming the most over-represented scenario and the net direction of movement.

Rows where MONTHLY OPPORTUNITY moved: 0 of 37,830 (0.0%)


In [10]:
# Same view by tier, plus a plain-language auto-summary
if n_moved and ('CLINIC TIER_base' in merged.columns):
    tier_tab = pd.DataFrame({
        'moved_%': (merged.loc[merged['moved'], 'CLINIC TIER_base'].value_counts(normalize=True)*100).round(1),
        'overall_%': (merged['CLINIC TIER_base'].value_counts(normalize=True)*100).round(1),
    }).fillna(0)
    print("Which TIERS moved:")
    display(tier_tab)

# Auto-summary sentence
if n_moved and (SC + '_base') in merged.columns:
    top_scen = tab.sort_values('over_represented_x', ascending=False).index[0]
    x = tab.loc[top_scen, 'over_represented_x']
    direction = 'UP' if (merged['delta']>0).sum() >= (merged['delta']<0).sum() else 'DOWN'
    print(f"\nSUMMARY: {n_moved:,} rows moved, mostly {direction}. "
          f"Most over-represented scenario among the changes: '{top_scen}' "
          f"({x}x its overall share). If that matches the population your change targeted, "
          f"the fix is landing where intended.")